# Test polytopes' volume computation

## Librairies

In [9]:
%load_ext autoreload
%autoreload 2

import sys
import os

# Go to project root (adjust depth if needed)
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH = os.path.join(ROOT, "data")
sys.path.append(ROOT)

print("Added to path:", ROOT)

import torch
from torchvision import datasets, transforms
from torch.utils.data import Subset

from src.models.networks import FashionMLP_Large
from src.quantization.quantize import quantize_model
from src.shortcuts.shortcut_weights import *
from src.optim.build_polytopes import *
from src.optim.prune_constraints import *
from src.optim.compute_volumes import estimate_polytope_width


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Added to path: /Users/jeremiecabessa/Desktop/ROOT/Articles/Conference_Papers/2025_With_Jiri/code/ErrorVolumePolytopes


In [10]:
torch.manual_seed(42)

## Large MLP (FashionMNIST)

In [15]:
# -------------------- #
# Load models and data #
# -------------------- #

# Device
device = torch.device("gpu" if torch.cuda.is_available() 
                      else "mps" if torch.backends.mps.is_available() 
                      else "cpu")

# Model
model_name = "fashion_mlp_best.pth"
model_path = os.path.join(ROOT, "checkpoints", model_name)

model = FashionMLP_Large()
model.load_state_dict(torch.load(model_path))#['model_state'])
model.to(device).eval()

# qModel
qmodel = quantize_model(model, bits=4)
qmodel.to(device).eval()

# Dataset (and sample)
train_dataset = torch.load(
    DATA_PATH + "/fashionMNIST_correct_mlp.pt",
    weights_only=False
)

x_0, c = train_dataset[13]
print(x_0.min().item(), x_0.max().item()) # check bounds
x_0 = x_0.flatten().unsqueeze(0) # shape (1, input_dim)
x_0 = x_0.to(device) # important
print("Sample x_0 shape:", x_0.shape)
print("Models and dataset have been loaded.")

-1.0 1.0
Sample x_0 shape: torch.Size([1, 784])
Models and dataset have been loaded.


In [16]:
%%time

# --------------------------- #
# Compute volume of polytopes #
# --------------------------- #

print("*** Computing volumes of polytopes... ***\n\n")

nb_directions = 10

bounds = [(-1., 1.)]

A_correct, b_correct, A_both, b_both = build_two_class_polytopes(model, qmodel, x_0, c)

results = estimate_polytope_width(A_correct, b_correct, bounds, 
                                    n_directions=nb_directions,
                                    verbose=True)
results

*** Computing volumes of polytopes... ***


Direction 0: w_correct=36.7948
Direction 1: w_correct=37.0233
Direction 2: w_correct=37.3783
Direction 3: w_correct=37.5208
Direction 4: w_correct=37.8128
Direction 5: w_correct=37.3328
Direction 6: w_correct=37.9290
Direction 7: w_correct=36.7647
Direction 8: w_correct=37.0145
Direction 9: w_correct=36.9750
CPU times: user 48.8 s, sys: 1.17 s, total: 50 s
Wall time: 50.3 s


{'mean_width_correct': 37.25460775879638,
 'std_width_correct': 0.3872054805251274,
 'n_directions_used': 10}

In [17]:
results = estimate_polytope_width(A_both, b_both, bounds, 
                                    n_directions=nb_directions,
                                    verbose=True)

Direction 0: w_correct=36.7660
Direction 1: w_correct=36.9072
Direction 2: w_correct=37.5994
Direction 3: w_correct=37.6570
Direction 4: w_correct=37.5098
Direction 5: w_correct=37.3481
Direction 6: w_correct=37.2809
Direction 7: w_correct=37.6659
Direction 8: w_correct=37.6282
Direction 9: w_correct=36.8465


In [18]:
torch.argmax(model(x_0)), torch.argmax(qmodel(x_0))

(tensor(7, device='mps:0'), tensor(7, device='mps:0'))